# Chapter 4: Training Fundamentals

## 100 Bad Guessers That Become One Great Predictor

We now have a clean numeric matrix (from Chapter 3). Time to teach a model to predict churn.

But before we hand data to XGBoost, we need to handle two critical setup problems:

1. **How do we split data fairly?** — The model needs a textbook (training), practice exams (validation), and a final exam (test). Each group must have the same proportion of churners as the original.

2. **What do we do about imbalance?** — If only 15% of customers churn, a lazy model can predict "won't churn" for everyone and be right 85% of the time. That's useless.

This chapter covers both problems, plus how we interpret predictions with risk tiers and SHAP explanations.

In [ ]:
import numpy as np
import pandas as pd
from churn_pipeline.steps.training import (
    create_stratified_splits,
    compute_scale_pos_weight,
    HYPERPARAMETER_RANGES,
)
from churn_pipeline.steps.scoring import (
    assign_risk_tier,
    extract_top_reasons,
    format_predictions,
)

## Part 1: Stratified Splitting

### The Problem

Imagine a bag with 80 blue marbles and 20 red marbles. If you grab a handful randomly, you might get 15 blue and 0 red — that handful doesn't represent the bag.

Same with customer data. If 20% of customers churned overall, each split (train/validation/test) should have ~20% churners. Otherwise:
- Training on 30% churners, testing on 10% churners → performance estimate is unreliable
- The model sees a different "world" in each split

**Stratified splitting** guarantees each group looks like the original.

In [ ]:
# Create a realistic dataset: 1000 customers, 20% churned
np.random.seed(42)
n_customers = 1000
n_features = 6

features = np.random.rand(n_customers, n_features)
labels = np.array([0] * 800 + [1] * 200)  # 20% churn rate
np.random.shuffle(labels)

print(f"Full dataset: {n_customers} customers, {labels.mean():.1%} churn rate")
print(f"Feature matrix shape: {features.shape}")

In [ ]:
# Split into train (70%), validation (15%), test (15%)
(X_train, y_train), (X_val, y_val), (X_test, y_test) = create_stratified_splits(
    features, labels
)

print(f"{'Split':<12} {'Size':<8} {'Churn Rate':<12} {'Preserved?'}")
print("-" * 45)
print(f"{'Original':<12} {n_customers:<8} {labels.mean():.3f}{'':>8}")
print(f"{'Train':<12} {len(y_train):<8} {y_train.mean():.3f}{'':>8} ✓")
print(f"{'Validation':<12} {len(y_val):<8} {y_val.mean():.3f}{'':>8} ✓")
print(f"{'Test':<12} {len(y_test):<8} {y_test.mean():.3f}{'':>8} ✓")
print(f"\nTotal samples: {len(y_train) + len(y_val) + len(y_test)} (no data lost)")

### Why 70/15/15?

| Split | Purpose | Analogy |
|-------|---------|--------|
| Train (70%) | The model reads this to learn patterns | The textbook |
| Validation (15%) | Checked during training to prevent memorization | Practice exams |
| Test (15%) | Final honest evaluation — never seen during training | The final exam |

The validation set is how we know when to stop training. Without it, the model would memorize every customer in the training set ("overfitting") instead of learning general patterns.

## Part 2: Handling Class Imbalance

### The Lazy Model Problem

If 85% of customers stay, a model can achieve 85% accuracy by always predicting "won't churn." That's useless — it catches zero actual leavers.

`scale_pos_weight` fixes this. It tells XGBoost: "Each churner counts as N non-churners in the loss function." The model pays N× more attention to getting churners right.

In [ ]:
# With our 80/20 split
weight = compute_scale_pos_weight(labels)
print(f"Dataset: {800} stayed, {200} left")
print(f"scale_pos_weight = {800}/{200} = {weight:.2f}")
print(f"\nMeaning: each churner counts as {weight:.1f} customers in the loss function.")
print(f"The model pays {weight:.1f}× more attention to getting churners right.")

In [ ]:
# What happens with different imbalance levels?
print(f"{'Churn Rate':<12} {'Stayed':<10} {'Left':<10} {'Weight':<10} {'Interpretation'}")
print("-" * 70)

for churn_pct in [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50]:
    n = 1000
    n_churn = int(n * churn_pct)
    test_labels = np.array([0] * (n - n_churn) + [1] * n_churn)
    w = compute_scale_pos_weight(test_labels)
    interp = f"Each churner counts as {w:.1f}×" if w > 1 else "No adjustment needed"
    print(f"{churn_pct:<12.0%} {n - n_churn:<10} {n_churn:<10} {w:<10.2f} {interp}")

Notice: when churn rate hits 30% or above, we stop applying the weight (returns 1.0). At that point the dataset is balanced enough that the model doesn't need extra help.

## Part 3: XGBoost Hyperparameters

XGBoost builds a team of decision trees. Each tree looks at the mistakes of the previous trees and tries to correct them. The hyperparameters are the "settings" that control how this team works.

Think of it like cooking: you're trying different oven temperatures, cook times, and ingredient ratios to find the perfect recipe. The tuner tries many combinations automatically.

In [ ]:
print("XGBoost Hyperparameter Ranges")
print("=" * 70)
for name, config in HYPERPARAMETER_RANGES.items():
    print(f"\n  {name} ({config['type']}, range: {config['min']} to {config['max']})")
    print(f"    {config['description']}")

## Part 4: From Predictions to Risk Tiers

After training, the model produces a probability for each customer (0.0 to 1.0). But business users don't want to interpret `0.73` — they want to know: **"Should I act NOW or later?"**

Risk tiers convert the raw number into an actionable label:

| Tier | Threshold | Action |
|------|-----------|--------|
| **High** | >= 0.7 | Intervene NOW — this customer is very likely leaving |
| **Medium** | 0.4 to 0.7 | Monitor closely — at risk, consider reaching out |
| **Low** | < 0.4 | Safe for now — no immediate action needed |

In [ ]:
# Demonstrate risk tier assignment
test_probs = [0.92, 0.85, 0.70, 0.55, 0.40, 0.39, 0.20, 0.05]

print(f"{'Probability':<14} {'Risk Tier':<12} {'Action'}")
print("-" * 55)
for p in test_probs:
    tier = assign_risk_tier(p)
    actions = {
        "high": "Intervene immediately",
        "medium": "Monitor, consider outreach",
        "low": "No action needed",
    }
    print(f"{p:<14.2f} {tier:<12} {actions[tier]}")

## Part 5: SHAP Explanations — "Show Your Work"

When the model says "Customer X: 85% chance of leaving," the client asks: **WHY?**

SHAP (SHapley Additive exPlanations) answers by computing how much each feature pushed the prediction up or down. It's like a jury explaining their verdict:

- "month-to-month contract added +23% risk" (pushed toward churn)
- "low tenure (2 months) added +15% risk" (pushed toward churn)  
- "having a partner reduced risk by -8%" (pushed away from churn)

We take the top 3 by absolute magnitude — the most influential factors, regardless of direction.

In [ ]:
# Simulate SHAP values for a high-risk customer
feature_names = ["tenure_months", "monthly_charges", "total_charges",
                  "contract_type", "support_tickets", "interaction_charges_x_tenure"]

# Customer with high churn probability
shap_high_risk = np.array([0.15, 0.08, -0.03, 0.23, 0.18, 0.05])

reasons = extract_top_reasons(shap_high_risk, feature_names, top_n=3)

print("High-risk customer — Top 3 reasons:")
print("=" * 45)
for i, reason in enumerate(reasons, 1):
    print(f"  {i}. {reason}")

print("\nTranslation:")
print("  contract_type (+0.23) → month-to-month contract is the biggest risk factor")
print("  support_tickets (+0.18) → frequent support calls signal frustration")
print("  tenure_months (+0.15) → very short tenure, hasn't built loyalty")

In [ ]:
# Low-risk customer — notice the negative (protective) factors
shap_low_risk = np.array([-0.22, -0.05, -0.12, -0.31, 0.02, -0.15])

reasons = extract_top_reasons(shap_low_risk, feature_names, top_n=3)

print("Low-risk customer — Top 3 reasons:")
print("=" * 45)
for i, reason in enumerate(reasons, 1):
    print(f"  {i}. {reason}")

print("\nTranslation:")
print("  contract_type (-0.31) → long-term contract locks them in")
print("  tenure_months (-0.22) → been a customer for years")
print("  interaction (-0.15) → high lifetime value")

## Part 6: The Final Deliverable

Everything comes together in the output CSV — what the client actually receives. One row per customer with:
- Who they are (`customer_id`)
- How likely they are to leave (`churn_probability`)
- The human-friendly label (`risk_tier`)
- The top 3 reasons why (`top_3_reasons`)
- A plain-English explanation (`narrative_explanation`)

In [ ]:
# Build a sample output
customer_ids = ["CUST_001", "CUST_002", "CUST_003", "CUST_004"]
probabilities = np.array([0.87, 0.55, 0.34, 0.92])
shap_matrix = np.array([
    [0.15, 0.08, -0.03, 0.23, 0.18, 0.05],   # high risk
    [0.10, 0.12, -0.05, 0.08, 0.06, -0.03],   # medium risk
    [-0.22, -0.05, -0.12, -0.31, 0.02, -0.15], # low risk
    [0.20, 0.15, 0.05, 0.25, 0.22, 0.10],     # very high risk
])

narratives = [
    "This customer has no long-term commitment and contacts support frequently.",
    "Moderate risk due to rising charges and short contract.",
    "N/A - low risk customer",
    "Extremely high risk: new customer, month-to-month, high support volume.",
]

output_df = format_predictions(
    customer_ids, probabilities, shap_matrix, feature_names, narratives
)

print("Client Deliverable (predictions.csv):")
print("=" * 80)
for _, row in output_df.iterrows():
    print(f"\n  {row['customer_id']} | prob={row['churn_probability']} | tier={row['risk_tier']}")
    print(f"    Reasons: {row['top_3_reasons']}")
    print(f"    Narrative: {row['narrative_explanation']}")

## Key Takeaways

1. **Stratified splitting** preserves class balance across train/val/test — each group looks like the whole
2. **scale_pos_weight** prevents lazy predictions by making the model pay extra attention to the rare class
3. **Risk tiers** convert probabilities into actionable labels (high/medium/low)
4. **SHAP** shows the model's work — the top factors pushing each prediction up or down
5. **The output CSV** combines everything into one client-ready deliverable

Next: Chapter 5 covers the evaluation gate — how we decide if a model is good enough to serve to clients.

---

*Source code: `src/churn_pipeline/steps/training.py`, `src/churn_pipeline/steps/scoring.py`*  
*Tests: `tests/property/test_training.py`, `tests/property/test_scoring.py`*  
*Series: [Build with AWS](https://buildwithaws.substack.com/)*